# ML-07 — Baseline Action Score and Top-10 Review

[Open in Colab](https://colab.research.google.com/github/Kifayat-Sayed/kifayat-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Goal:** test two plausible signals, encode one transparent rule, write a ranked queue, and review the top ten skeptically.

**Unit:** one anonymized content item in the starter snapshot. Signal audits use the observed decline label for descriptive checks only. The rule does **not** use the label, `trend_direction`, `trend_pct`, or future-window fields.

## 0. Load the starter dataset

This cell uses the repository's anonymized starter CSV. If the repo is not already available in the runtime, it clones it. The generated queue is written to `work/outputs/baseline_action_score.csv`; the dataset and generated CSV should not be committed.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Kifayat-Sayed/kifayat-flyrank-ml-internship.git"
ROOT = Path.cwd()
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

if not (ROOT / DATA_REL).exists():
    clone_dir = ROOT / "kifayat-flyrank-ml-internship"
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    ROOT = clone_dir

DATA_PATH = ROOT / DATA_REL
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Open the notebook from the internship repo "
        "or confirm that data/raw/content_refresh_anonymized.csv is available."
    )

data = pd.read_csv(DATA_PATH)
required = {"content_id", "days_since_last_update", "impressions_90d", "trend_direction"}
missing = required - set(data.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

# Audit-only target. Never use this column to calculate score or action.
if "is_declining_label" in data.columns:
    data["_audit_declining"] = pd.to_numeric(data["is_declining_label"], errors="coerce")
else:
    data["_audit_declining"] = data["trend_direction"].astype(str).str.lower().eq("down").astype(int)

print(f"Loaded {len(data):,} rows × {data.shape[1]} columns")
print(f"Observed decline-label prevalence: {data['_audit_declining'].mean():.1%} "
      f"(n={data['_audit_declining'].notna().sum():,})")
print("Rule inputs:", ["days_since_last_update", "impressions_90d"])

Loaded 30,000 rows × 45 columns
Observed decline-label prevalence: 54.2% (n=30,000)
Rule inputs: ['days_since_last_update', 'impressions_90d']


## 1. Two signal checks and verdicts

**Signal A — staleness:** pages that have gone longer without an update may be more likely to decline. This is connected to the refresh-review flags discussed in the session.

**Signal B — impression volume:** higher-volume pages may be more useful quick-win candidates because a change could affect a larger observed search audience.

Each bucket table prints `n` and the observed decline rate. Verdicts are based on the pattern across buckets with at least 50 labelled rows. A negative or mixed result is a valid finding, not a failed assignment.

In [2]:
# Bucket boundaries are explicit and easy to inspect.
data["staleness_bucket"] = pd.cut(
    pd.to_numeric(data["days_since_last_update"], errors="coerce"),
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0–30 days", "31–90 days", "91–180 days", "181+ days"],
    include_lowest=True
)
data["volume_bucket"] = pd.cut(
    pd.to_numeric(data["impressions_90d"], errors="coerce"),
    bins=[-np.inf, 299, 2999, 29999, np.inf],
    labels=["<300", "300–2,999", "3,000–29,999", "30,000+"],
    include_lowest=True
)

def bucket_table(bucket_col):
    grouped = data.groupby(bucket_col, observed=False)["_audit_declining"]
    table = grouped.agg(n="size", labelled_n="count", decline_rate="mean").reset_index()
    table["decline_rate_pct"] = (100 * table["decline_rate"]).round(1)
    return table.drop(columns="decline_rate")

def signal_verdict(table, bucket_col):
    usable = table.loc[(table["labelled_n"] >= 50) & table["decline_rate_pct"].notna()].copy()
    if len(usable) < 2:
        return "MIXED", "Fewer than two buckets have at least 50 labelled rows; evidence is insufficient."
    rates = usable["decline_rate_pct"].astype(float).to_numpy()
    differences = np.diff(rates)
    endpoint = rates[-1] - rates[0]
    if np.all(differences >= -3) and endpoint >= 5:
        return "CONFIRMED", f"Broadly rising pattern; endpoint difference is {endpoint:+.1f} percentage points."
    if np.all(differences <= 3) and endpoint <= -5:
        return "OPPOSITE", f"Broadly falling pattern; endpoint difference is {endpoint:+.1f} percentage points."
    if rates.max() - rates.min() < 5:
        return "FALSE", "Rates vary by less than 5 percentage points across eligible buckets; no meaningful bucket signal is evident."
    return "MIXED", f"Pattern is not consistently monotonic; endpoint difference is {endpoint:+.1f} percentage points."

staleness_results = bucket_table("staleness_bucket")
print("SIGNAL A — DAYS SINCE LAST UPDATE vs observed decline")
display(staleness_results)
staleness_verdict, staleness_note = signal_verdict(staleness_results, "staleness_bucket")
print(f"VERDICT: {staleness_verdict} — {staleness_note}")

volume_results = bucket_table("volume_bucket")
print("\nSIGNAL B — 90-DAY IMPRESSIONS vs observed decline")
display(volume_results)
volume_verdict, volume_note = signal_verdict(volume_results, "volume_bucket")
print(f"VERDICT: {volume_verdict} — {volume_note}")

SIGNAL A — DAYS SINCE LAST UPDATE vs observed decline


,staleness_bucket,n,labelled_n,decline_rate_pct
0,0–30 days,20480,20480,51.1
1,31–90 days,175,175,58.9
2,91–180 days,9171,9171,61.1
3,181+ days,174,174,47.1


VERDICT: MIXED — Pattern is not consistently monotonic; endpoint difference is -4.0 percentage points.

SIGNAL B — 90-DAY IMPRESSIONS vs observed decline


,volume_bucket,n,labelled_n,decline_rate_pct
0,<300,11248,11248,45.4
1,"300–2,999",10469,10469,61.5
2,"3,000–29,999",7205,7205,58.6
3,"30,000+",1078,1078,46.2


VERDICT: MIXED — Pattern is not consistently monotonic; endpoint difference is +0.8 percentage points.


## 2. Encode one rule and write the ranked queue

**Rule in plain words:** prioritise a refresh review when a page has not been updated for at least 180 days and has at least 3,000 impressions in the observed 90-day snapshot. Rank qualifying pages by their impression volume.

**Score:** `stale_flag × visible_flag × impressions_90d`.

**Reason code:** `stale_high_visibility_rule` (one reason-code field applied consistently to the queue).

**Actions:** `refresh_review` or `monitor`.

These are hand-written thresholds, not learned weights. The score uses only `days_since_last_update` and `impressions_90d`. The audit label is deliberately excluded from ranking.

In [3]:
stale_days = pd.to_numeric(data["days_since_last_update"], errors="coerce")
impressions = pd.to_numeric(data["impressions_90d"], errors="coerce").fillna(0)

stale_flag = stale_days.ge(180).fillna(False).astype(int)
visible_flag = impressions.ge(3000).astype(int)

ranked = data.copy()
ranked["score"] = stale_flag * visible_flag * impressions
ranked["reason_code"] = "stale_high_visibility_rule"
ranked["action"] = np.where((stale_flag == 1) & (visible_flag == 1),
                            "refresh_review", "monitor")

# IDs are used only to identify rows and break ties deterministically, never as score inputs.
ranked = ranked.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    kind="mergesort"
).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

out_dir = ROOT / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"
export_cols = [
    "rank", "content_id", "score", "reason_code", "action",
    "days_since_last_update", "impressions_90d", "avg_position", "ctr"
]
export_cols = [col for col in export_cols if col in ranked.columns]
ranked[export_cols].to_csv(out_path, index=False)

print(f"CSV written: {out_path}")
print(f"Queue rows: {len(ranked):,}")
print(f"refresh_review: {(ranked['action'] == 'refresh_review').sum():,}")
print(f"monitor: {(ranked['action'] == 'monitor').sum():,}")
display(ranked[export_cols].head(10))

CSV written: /content/kifayat-flyrank-ml-internship/work/outputs/baseline_action_score.csv
Queue rows: 30,000
refresh_review: 9
monitor: 29,991


,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_cf56e2e2e282,61678,stale_high_visibility_rule,refresh_review,194,61678,19.7,0.15
1,2,content_7368877ea310,59472,stale_high_visibility_rule,refresh_review,194,59472,24.8,0.13
2,3,content_1bfaa38ff26c,25715,stale_high_visibility_rule,refresh_review,194,25715,22.2,0.23
3,4,content_0a91db491d14,13299,stale_high_visibility_rule,refresh_review,193,13299,10.5,0.49
4,5,content_5feee3994adb,7812,stale_high_visibility_rule,refresh_review,194,7812,39.0,0.01
5,6,content_c2d929d83eaa,7558,stale_high_visibility_rule,refresh_review,193,7558,17.9,0.20
6,7,content_b16bd7307b39,4590,stale_high_visibility_rule,refresh_review,194,4590,31.0,0.00
7,8,content_fe16a55cd13d,4556,stale_high_visibility_rule,refresh_review,194,4556,16.4,0.33
8,9,content_ecb6215e79fd,4429,stale_high_visibility_rule,refresh_review,194,4429,25.3,0.38
9,10,content_5fe46e04994d,0,stale_high_visibility_rule,monitor,104,517715,4.2,0.14


## 3. Top-10 skeptical review

For each ranked item, the notebook prints its action, reason code, why it is in that position, a confidence note, and what could make the recommendation wrong. These are triage hypotheses, not proof that refreshing causes improvement.

In [4]:
top10 = ranked.head(10).copy()

def explain_top_row(row):
    action = row["action"]
    days = row["days_since_last_update"]
    imps = row["impressions_90d"]
    if action == "refresh_review":
        why = (f"Meets both rule thresholds: last update {days:g} days ago and "
               f"{imps:,.0f} impressions; score={row['score']:,.0f}.")
        confidence = "Moderate rule confidence; confirm page intent and current performance before acting."
    else:
        why = (f"Does not meet both thresholds (last update={days if pd.notna(days) else 'unknown'} days; "
               f"impressions={imps:,.0f}); score={row['score']:,.0f}.")
        confidence = "Low action priority under this rule; monitor does not mean the page is healthy."
    wrong = ("The update date may be inaccurate, traffic may be seasonal, the page may be intentionally "
             "evergreen, or current rankings/CTR may already be healthy.")
    pos = pd.to_numeric(pd.Series([row.get("avg_position", np.nan)]), errors="coerce").iloc[0]
    ctr = pd.to_numeric(pd.Series([row.get("ctr", np.nan)]), errors="coerce").iloc[0]
    if pd.notna(pos) and pos > 0 and pos <= 10:
        wrong = ("Average position is already on page one; staleness alone may not justify a refresh. "
                 "Check query intent, CTR, and recent changes.")
    elif pd.notna(pos) and pos > 10 and pd.notna(ctr) and ctr > 1:
        wrong = ("CTR may already be healthy for its position, or impressions may be seasonal; "
                 "check query mix and recent changes before acting.")
    return pd.Series({
        "rank": int(row["rank"]),
        "content_id": row["content_id"],
        "action": action,
        "reason_code": row["reason_code"],
        "why_it_is_here": why,
        "confidence_note": confidence,
        "what_would_make_it_wrong": wrong
    })

top10_review = top10.apply(explain_top_row, axis=1)
for _, r in top10_review.iterrows():
    print(f"{int(r['rank'])}. {r['content_id']} — ACTION: {r['action']}; "
          f"REASON: {r['reason_code']}; WHY: {r['why_it_is_here']} "
          f"CONFIDENCE: {r['confidence_note']} WHAT COULD MAKE IT WRONG: {r['what_would_make_it_wrong']}")

display(top10_review)

1. content_cf56e2e2e282 — ACTION: refresh_review; REASON: stale_high_visibility_rule; WHY: Meets both rule thresholds: last update 194 days ago and 61,678 impressions; score=61,678. CONFIDENCE: Moderate rule confidence; confirm page intent and current performance before acting. WHAT COULD MAKE IT WRONG: The update date may be inaccurate, traffic may be seasonal, the page may be intentionally evergreen, or current rankings/CTR may already be healthy.
2. content_7368877ea310 — ACTION: refresh_review; REASON: stale_high_visibility_rule; WHY: Meets both rule thresholds: last update 194 days ago and 59,472 impressions; score=59,472. CONFIDENCE: Moderate rule confidence; confirm page intent and current performance before acting. WHAT COULD MAKE IT WRONG: The update date may be inaccurate, traffic may be seasonal, the page may be intentionally evergreen, or current rankings/CTR may already be healthy.
3. content_1bfaa38ff26c — ACTION: refresh_review; REASON: stale_high_visibility_rule; WHY: M

,rank,content_id,action,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
1,2,content_7368877ea310,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
2,3,content_1bfaa38ff26c,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
3,4,content_0a91db491d14,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 193 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
4,5,content_5feee3994adb,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
5,6,content_c2d929d83eaa,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 193 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
6,7,content_b16bd7307b39,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
7,8,content_fe16a55cd13d,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
8,9,content_ecb6215e79fd,refresh_review,stale_high_visibility_rule,Meets both rule thresholds: last update 194 da...,Moderate rule confidence; confirm page intent ...,"The update date may be inaccurate, traffic may..."
9,10,content_5fe46e04994d,monitor,stale_high_visibility_rule,Does not meet both thresholds (last update=104...,Low action priority under this rule; monitor d...,Average position is already on page one; stale...


## 4. Weak picks and leakage check

Inspect high-visibility pages left on `monitor`: they are potential false negatives if other evidence indicates a refresh is needed. Also inspect any selected page with healthy current performance before taking action.

In [5]:
weak_picks = ranked[
    (ranked["action"] == "monitor") &
    (pd.to_numeric(ranked["impressions_90d"], errors="coerce").fillna(0) >= 3000)
].sort_values("impressions_90d", ascending=False).head(5)

print("Potential weak picks: visible pages left on monitor because they did not meet the staleness threshold.")
if len(weak_picks):
    display(weak_picks[[c for c in [
        "content_id", "action", "days_since_last_update",
        "impressions_90d", "avg_position", "ctr"
    ] if c in weak_picks.columns]])
    print("Why they may be weak picks: the fixed 180-day cutoff can miss a page that needs attention for reasons other than staleness.")
else:
    print("No high-visibility monitor rows found in this snapshot.")

# Explicitly verify that no label-derived or trend-window field enters the rule.
rule_inputs = {"days_since_last_update", "impressions_90d"}
forbidden_inputs = {"trend_pct", "trend_direction", "is_declining_label", "_audit_declining",
                    "impressions_last_30d", "impressions_prev_30d"}
assert rule_inputs.isdisjoint(forbidden_inputs)
assert set(ranked["reason_code"].unique()) == {"stale_high_visibility_rule"}
assert set(ranked["action"].unique()).issubset({"refresh_review", "monitor"})
assert out_path.exists() and out_path.stat().st_size > 0
assert len(top10_review) == min(10, len(ranked))
print("LEAKAGE CHECK: PASS — score and action use only days_since_last_update and impressions_90d.")
print("OUTPUT CHECK: PASS — required CSV exists and is non-empty.")
print("LIMITATION: the rule prioritises review; it does not establish that refreshing causes better outcomes.")

Potential weak picks: visible pages left on monitor because they did not meet the staleness threshold.


,content_id,action,days_since_last_update,impressions_90d,avg_position,ctr
9,content_5fe46e04994d,monitor,104,517715,4.2,0.14
10,content_aaef01a50def,monitor,22,517109,5.4,0.25
11,content_8c19996aa890,monitor,20,509252,2.5,0.15
12,content_2cb567c3c89b,monitor,48,497727,22.2,0.10
13,content_4c36c775b818,monitor,20,463103,2.3,0.41


Why they may be weak picks: the fixed 180-day cutoff can miss a page that needs attention for reasons other than staleness.
LEAKAGE CHECK: PASS — score and action use only days_since_last_update and impressions_90d.
OUTPUT CHECK: PASS — required CSV exists and is non-empty.
LIMITATION: the rule prioritises review; it does not establish that refreshing causes better outcomes.


## 5. Self-check

- [x] Two signal checks produce bucket tables with visible sample counts and one-word verdicts.
- [x] Staleness is linked to the session's refresh-review flag logic.
- [x] One hand-written score, one reason-code field, and action labels are defined.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Top ten are reviewed for action, reason, confidence, and what could make each recommendation wrong.
- [x] The rule excludes `trend_pct`, `trend_direction`, and label-derived inputs.
- [ ] Run all cells from top to bottom and confirm no errors.
- [ ] Commit the executed notebook. Keep the generated CSV out of git.

**Interpretation guardrail:** signal verdicts describe associations in this snapshot only. They do not prove causality or guarantee that this baseline will be beaten; Week 5's model should be compared on the same slice and label definition.